# HA#5 - Sarcasm Detection with DistilBERT
**CSC620 Natural Language Processing**
**Author:** Ryan Alvarado

---

## What are we doing, and why?
HA#3 was built on Naïve Bayes classifier. It works by counting word frequencies and using Bayes' theorem to estimate the probability that a headline is sarcastic. It treats every word independently with no concept of context.

In this project we are using DistilBERT, a transformer-based language model. Transformers read every word in relation to every other word in the sentence. This model can pick up on subtle cues such as irony, tone, and contrast.

Fine-tuning means we start from a model that already knows English and train it for a short time on our specific task.

## Concepts used
**Transformer embeddings**
Each token is represented as a dense vector of numbers that encode meaning and context

**CLS token**
A special token prepended to every input. After passing through all transformer layers its embedding summarises the entire sentence.

**Classification head**
A simple linear layer (of matrix multiplication) that maps the CLS vector to one score per class

**Fine-tuning**
Continuing training on task-specific data so the pre-training weights specialize to our problem

### Data Flow
```
Raw headline text
        |
        v
Tokenizer  ->  token IDs + attention mask
        |
        v
DistilBERT (6 transformer layers)  ->  contextual embedding for every token
        |
        v
[CLS] token embedding  ->  single vector summarising the whole sentence
        |
        v
Classification head (Linear 768 -> 2)  ->  logits [score_0, score_1]
        |
        v
argmax  ->  predicted label (0 = not sarcastic, 1 = sarcastic)
```

---
## HA #5 pt. 2: Fine-Tuning DistilBERT for Sarcasm Detection

### Step 1: Install Dependencies

In [ ]:
# These packages are not pre-installed in a fresh Colab environment.
#
# transformers  - Hugging Face library that provides DistilBERT and the Trainer API
# datasets      - Hugging Face library for easy dataset loading (not used directly here but good to have)
# scikit-learn  - provides train_test_split and all evaluation metrics
# torch         - PyTorch, the deep learning framework DistilBERT runs on
#
# --quiet suppresses the verbose install output so the cell stays readable

!pip install transformers datasets scikit-learn torch --quiet

## Imports

In [ ]:
import json         # to read .json dataset
import pandas as pd # for tabular data manipulation & display
import numpy as np  # for numerical operations (argmax on logits, etc.)

# train_test_split: randomly divides data into a training set and a held-out test set
from sklearn.model_selection import train_test_split

# Accuracy  - overall fraction correct (can be misleading on bad data)
# Precision - of everything the model labeled sarcastic, how many truly were?
# Recall    - of all the truly sarcastic headlines, how many did the model catch?
# F1        - harmonic mean of precision and recall, balances both concerns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Dataset - abstract PyTorch class; we subclass it to define how to fetch one example
import torch
from torch.utils.data import Dataset

# DistilBertTokenizerFast             - converts raw text into token IDs the model reads
# DistilBertForSequenceClassification - DistilBERT backbone + classification head
# Trainer / TrainingArguments         - Hugging Face high-level training loop
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

### Step 3: Load the Dataset

**Before running this cell:**
1. Download `Sarcasm_Headlines_Dataset.json` from https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection
2. Upload it to your Colab session using the Files panel on the left sidebar.

In [ ]:
# The file is newline-delimited JSON. Each line is a complete JSON object.
# We can't use json.load() on the whole file at once, so we go line-by-line.

records = []
with open('Sarcasm_Headlines_Dataset.json', 'r') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

# DataFrame has three columns:
#   article_link  - URL to the original article (ignored)
#   headline      - news headline text (our input X)
#   is_sarcastic  - 1 = sarcastic, 0 = non-sarcastic (label y)

print(df.head())
print(f"\nTotal samples: {len(df)}")
print(f"Sarcastic: {df['is_sarcastic'].sum()}")
print(f"Not sarcastic: {(df['is_sarcastic'] == 0).sum()}")

## Train / Test Split

In [ ]:
# Pull text and labels out of the DataFrame as plain Python list.
# The tokenizer and Dataset class expects lists, not pandas Series.

texts  = df['headline'].tolist()
labels = df['is_sarcastic'].tolist()

# Split 80% train / 20% test
# random_state=42 -> fixes the random seed so the split is reproducible every run
# stratify=labels -> both splits will have the same sarcastic/non-sarcastic ratio,
#                    which matters if the classes are not perfectly balanced

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

print(f"Training examples : {len(X_train)}")
print(f"Test examples     : {len(X_test)}")

## Tokenize

Transformers can't read raw strings. The tokenizer converts headlines into:
- **input_ids** - a list of integers, one per sub-word token (playing → "play" + "##ing")
- **attention_mask** - 1 for real tokens, 0 for padding tokens

GPUs process in batches, so padding is necessary. All sequences must be the same length to form tensors.

In [ ]:
# Load tokenizer that pairs with distilbert-base-uncased:
# distilbert - a compressed version of BERT, ~40% fewer parameters
# base       - standard size (not the larger 'large' variant)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Tokenize the entire training and test sets at once (batch tokenization is faster)
# * cut off headlines longer than max_length
# * pad shorter headlines with [PAD] tokens up to the longest
# * 128 tokens is generous for news headlines

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(X_test,  truncation=True, padding=True, max_length=128)

# Sanity check: decode the first training headline to confirm tokenizer round-trips
example_ids = train_encodings['input_ids'][0]
print("Token IDs for first training headline:")
print(example_ids)
print("\nDecoded back to text:")
print(tokenizer.decode(example_ids))

## Concept check
### What does the base model see?

According to the blog, the pre-trained base model cannot distinguish emotionally and tonally different sentences yet. It was trained on general language, so its embeddings cluster all grammatically similar sentences close together.

We verify using cosine similarity on [CLS] token embeddings of two headlines: one sarcastic, one not.

- Cosine similarity = 1.0 means the vectors point in the same direction — the model thinks they are identical.
- 0.0 means perpendicular, or unrelated.
- Negative means opposite.

If the base model understood sarcasm, a sarcastic and a genuine headline should have low similarity:

In [ ]:
from transformers import AutoModel
from torch.nn.functional import cosine_similarity

# Load the raw base model - NO classification head, just the transformer backbone.
# This is what the model looks like before we fine-tune it for sarcasm.
base_model = AutoModel.from_pretrained('distilbert-base-uncased')

def get_cls_embedding(text, model):
    """
    Tokenize 'text', run it through 'model', and return the [CLS] token embedding.

    The [CLS] token is always position 0 in the output.
    Its embedding is a 768-dimensional vector summarising the whole sentence.
    last_hidden_state shape: (batch=1, sequence_length, hidden_size=768)
    We want [0][0] -> first batch item, first token [CLS].

    Note: model is passed as a parameter so this function works for both
    the base (untrained) model and the fine-tuned model backbone.
    """
    inputs = tokenizer(text, return_tensors='pt')   # return PyTorch tensors
    with torch.no_grad():                           # no gradients needed for inference
        output = model(**inputs).last_hidden_state  # use the passed-in model, not base_model
    return output[0][0]                             # [CLS] embedding vector

# Pick one real sarcastic headline & 1 genuine headline from dataset
sarcastic_example     = "area man passionate defender of what he calls 'real' country music"
not_sarcastic_example = "country music fans gather for annual festival in nashville"

emb_sarc = get_cls_embedding(sarcastic_example,     base_model)
emb_real = get_cls_embedding(not_sarcastic_example, base_model)

# cosine_similarity expects 2D tensors, so we add a batch dimension with unsqueeze(0)
sim_before = cosine_similarity(emb_sarc.unsqueeze(0), emb_real.unsqueeze(0)).item()

print(f"Sarcastic : {sarcastic_example}")
print(f"Genuine   : {not_sarcastic_example}")
print(f"\nCosine similarity BEFORE fine-tuning: {round(sim_before, 4)}")
print()
print("A score close to 1.0 means the base model sees these as nearly identical.")
print("This is the problem fine-tuning fixes.")

### Step 6: Build a PyTorch Dataset

Hugging Face's `Trainer` expects a `Dataset` object that returns **one example at a time** by index. We subclass `torch.utils.data.Dataset` and implement two required methods:
- `__len__`     - how many examples total?
- `__getitem__` - give me example number `idx` as a dict of tensors

Think of this class as a bridge between our tokenized lists and PyTorch's batching machinery.

In [ ]:
class SarcasmDataset(Dataset):
    """
    Wraps tokenized encodings and integer labels into a format
    the Hugging Face Trainer can iterate over batch-by-batch.
    """

    def __init__(self, encodings, labels):
        # encodings is a dict of lists, e.g.:
        #   {'input_ids': [[101, 2054, ...], ...], 'attention_mask': [[1, 1, ...], ...]}
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        # Required by PyTorch so it knows how many steps make one epoch
        return len(self.labels)

    def __getitem__(self, idx):
        # Grab the idx-th element from every encoding list and wrap it in a tensor.
        # The Trainer will call this for each example in a batch.
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        # The Trainer needs 'labels' in the returned dict to compute cross-entropy loss
        item['labels'] = torch.tensor(self.labels[idx])
        return item


train_dataset = SarcasmDataset(train_encodings, y_train)
test_dataset  = SarcasmDataset(test_encodings,  y_test)

print(f"Train dataset : {len(train_dataset)} examples")
print(f"Test dataset  : {len(test_dataset)} examples")
print("\nOne example (keys and tensor shapes):")
for k, v in train_dataset[0].items():
    print(f"  {k:15s} shape={tuple(v.shape)}, dtype={v.dtype}")

### Step 7: Load the Model and Configure Training

`DistilBertForSequenceClassification` is the DistilBERT backbone with a **classification head** bolted on:

```
Input token IDs
        |
DistilBERT (6 transformer layers - these weights come pre-trained)
        |
[CLS] token embedding  <-- summarises the full sentence in one 768-dim vector
        |
Dropout  <-- randomly zeroes some values during training to prevent overfitting
        |
Linear(768 -> 2)  <-- the classification head; only these weights start from scratch
        |
Logits: [score_not_sarcastic, score_sarcastic]
```

During fine-tuning ALL weights are updated, but the transformer layers barely change (small learning rate). The classification head changes the most since it starts randomly initialised.

In [ ]:
# Load DistilBERT pre-trained weights and attach a 2-class classification head.
# num_labels=2 tells the model: output two logits per example (one per class).

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2,
)

# TrainingArguments is a configuration object - it does NOT start training.
# Think of it as the settings panel before you hit 'Run'.

training_args = TrainingArguments(
    output_dir='./results',             # folder where checkpoints are saved after each epoch
    num_train_epochs=3,                 # how many full passes through the training data
    per_device_train_batch_size=16,     # examples per gradient update; larger = faster but uses more RAM
    per_device_eval_batch_size=32,      # examples per eval step; no gradients so we can go larger
    evaluation_strategy='epoch',        # run evaluation on the test set at the end of every epoch
    save_strategy='epoch',              # save a checkpoint at the end of every epoch
    load_best_model_at_end=True,        # when training finishes, restore the checkpoint with best eval score
    logging_dir='./logs',               # TensorBoard log directory
    logging_steps=50,                   # print a progress line every 50 gradient steps
    report_to='none',                   # disable Weights & Biases logging (prevents login prompt in Colab)
)

print("Model and training arguments ready.")

### Step 8: Define Metrics and Run Training

The `Trainer` calls `compute_metrics` automatically after each evaluation epoch. It receives the raw **logits** (unnormalised scores) and the true labels, and we return a dict of whatever metrics we want to track.

In [ ]:
def compute_metrics(eval_pred):
    """
    Receives model outputs and ground-truth labels after each eval epoch.

    Arguments
    ---------
    eval_pred : (logits, labels)
        logits - numpy array of shape (n_examples, 2)
                 raw scores output by the classification head
        labels - numpy array of shape (n_examples,)
                 true 0/1 labels for each example
    """
    logits, labels = eval_pred

    # Convert logits to predicted class labels.
    # argmax picks the index of the highest score along the last axis.
    # Example: logits = [[-1.2,  2.4],   -> preds = [1, 0]
    #                    [ 0.8, -0.3]]
    preds = np.argmax(logits, axis=-1)

    return {
        'accuracy' : accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall'   : recall_score(labels, preds, zero_division=0),
        'f1'       : f1_score(labels, preds, zero_division=0),
    }


# The Trainer owns the entire training loop:
# forward pass -> loss computation -> backward pass -> weight update -> repeat
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# TRAINING TIP: In Colab go to Runtime -> Change runtime type -> T4 GPU
# to cut training time from ~45 minutes (CPU) down to ~5 minutes (GPU).
print("Starting training...")
trainer.train()

In [ ]:
# ── After fine-tuning: did the [CLS] embeddings actually separate? ────────────
#
# The blog's key insight: fine-tuning should push sarcastic and non-sarcastic
# headlines APART in the embedding space, not just train the classification head.
# We verify this by re-running the same cosine similarity check from before training.
#
# model.distilbert gives us the backbone without the classification head,
# so the embedding shapes are the same as the base_model used above.

emb_sarc_ft = get_cls_embedding(sarcastic_example,     model.distilbert)
emb_real_ft = get_cls_embedding(not_sarcastic_example, model.distilbert)

sim_after = cosine_similarity(emb_sarc_ft.unsqueeze(0), emb_real_ft.unsqueeze(0)).item()

print(f"Cosine similarity BEFORE fine-tuning : {round(sim_before, 4)}")
print(f"Cosine similarity AFTER  fine-tuning : {round(sim_after,  4)}")
print()
print("The similarity should drop significantly (or go negative) after training.")
print("This means the model learned to represent sarcastic vs. genuine headlines")
print("as genuinely different directions in 768-dimensional space.")

### Step 9: Evaluate and Save Submission Files

After training, we run a final inference pass on the test set to get predictions, print a detailed classification report, and write the two required submission files.

In [ ]:
# trainer.predict() runs a forward pass on every example in test_dataset.
# No gradients are computed (equivalent to torch.no_grad()), so it's fast.
predictions = trainer.predict(test_dataset)

# predictions.predictions has shape (n_test, 2).
# We argmax across axis=-1 to turn each pair of logits into a single 0 or 1.
preds = np.argmax(predictions.predictions, axis=-1)

# classification_report prints per-class and overall precision / recall / F1.
# 'support' is just the count of true examples in each class.
report = classification_report(
    y_test, preds,
    target_names=['Not Sarcastic', 'Sarcastic'],
)
print("=== DistilBERT Classification Report ===")
print(report)

# ── classification_report.txt ─────────────────────────────────────────────────
# Submission file that holds both the DistilBERT report and the HA3 NB results.
with open('classification_report.txt', 'w') as f:
    f.write("=== DistilBERT Classification Report ===\n\n")
    f.write(report)
    f.write("\n\n=== Naive Bayes Results (from HA3) ===\n\n")
    f.write("# Paste your HA3 classification report output here\n")

print("Saved -> classification_report.txt")

# ── sarcasm_predictions.csv ───────────────────────────────────────────────────
# Each row shows the headline, the correct answer, and what the model predicted.
# Rows where true_label != predicted_label are the model's mistakes - good for error analysis.
pred_df = pd.DataFrame({
    'headline'        : X_test,
    'true_label'      : y_test,
    'predicted_label' : preds,
})
pred_df.to_csv('sarcasm_predictions.csv', index=False)
print("Saved -> sarcasm_predictions.csv")

### Step 10: Compare DistilBERT vs. Naive Bayes (HA3)

Fill in the four Naive Bayes numbers from your HA3 output, then run the cell to see a side-by-side comparison table.

In [ ]:
# ── Paste your HA3 Naive Bayes results here ───────────────────────────────────
nb_accuracy  = 0.00   # e.g., 0.847
nb_precision = 0.00   # precision for the sarcastic class
nb_recall    = 0.00   # recall for the sarcastic class
nb_f1        = 0.00   # F1 for the sarcastic class

# ── Compute DistilBERT metrics from the predictions saved in Step 9 ───────────
db_accuracy  = accuracy_score(y_test, preds)
db_precision = precision_score(y_test, preds, zero_division=0)
db_recall    = recall_score(y_test, preds, zero_division=0)
db_f1        = f1_score(y_test, preds, zero_division=0)

# ── Build comparison table ────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Metric'     : ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Naive Bayes': [nb_accuracy, nb_precision, nb_recall, nb_f1],
    'DistilBERT' : [db_accuracy, db_precision, db_recall, db_f1],
})
comparison = comparison.set_index('Metric').round(4)

# Positive difference = DistilBERT is better on that metric
comparison['Difference (+/-)'] = comparison['DistilBERT'] - comparison['Naive Bayes']

print(comparison.to_string())

---
## Part 3: Reflection and Analysis

### A. Model Performance

**In what ways did DistilBERT outperform Naive Bayes on this task? Did you observe any shortcomings of the language model?**

*Write your response here (4-5 sentences).*

---

### B. Practical Considerations

**What are the tradeoffs between using Naive Bayes and a fine-tuned LLM? Which model would you choose for a real-world sarcasm detection system, and why?**

*Write your response here (4-5 sentences).*